In [11]:
import os
from pathlib import Path

# Project root (directory containing this script)
PROJECT_ROOT = Path.cwd().resolve().parent
print(f"Setting project root: {PROJECT_ROOT}")

# Tool paths
tool_paths = [
    PROJECT_ROOT / ".tools/cadical/build",
    PROJECT_ROOT / ".tools/abc",
    PROJECT_ROOT / ".tools/oss-cad-suite/bin",
]

# Update PATH (prepend like the shell script)
os.environ["PATH"] = ":".join(map(str, tool_paths)) + ":" + os.environ["PATH"]

print(f"Updated PATH to include: {tool_paths}")

# If you also want the variable exported like in bash
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
data_path = PROJECT_ROOT / "data"

Setting project root: /home/krishnendu/Research/fv-invariant-mining
Updated PATH to include: [PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadical/build'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/abc'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/oss-cad-suite/bin')]


## Utilities

In [5]:
# Write a function that takes a shell command, runs it and returns its exit code and output as a tuple.
import subprocess
def run_command(command):
    result = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return (result.returncode, result.stdout.decode('utf-8') + result.stderr.decode('utf-8'))

In [7]:
def verilog_to_aig(verilog_file, top_module, output_aig="output.aig"):
    """
    Compile a specific Verilog module to AIG using Yosys.

    Args:
        verilog_file (str): Path to the Verilog source file
        top_module (str): Name of the module to synthesize
        output_aig (str): Output AIG file name
    """

    if type(verilog_file) is str:
        try:
            verilog_file = str(verilog_file)
        except Exception as e:
            raise ValueError(f"Invalid verilog_file path: {verilog_file}") from e
        
    if type(output_aig) is str:
        try:
            output_aig = str(output_aig)
        except Exception as e:
            raise ValueError(f"Invalid output_aig path: {output_aig}") from e
        
    # Create all leading dirs to output_aig if they don't exist
    output_dir = os.path.dirname(output_aig)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    yosys_script = f"""
    read_verilog {verilog_file}
    hierarchy -check -top {top_module}
    proc
    opt
    techmap
    aigmap
    write_aiger {output_aig}
    """

    command = f'yosys -p "{yosys_script}"'
    exit_code, output = run_command(command)

    return exit_code, output

In [8]:
def aig_cec(aig1, aig2):
    """
    Run combinational equivalence checking (CEC) between two AIG files using ABC.

    Args:
        aig1 (str): Path to first AIG file
        aig2 (str): Path to second AIG file

    Returns:
        (exit_code, output)
    """

    abc_script = f"""
    read {aig1};
    cec {aig2}
    """

    command = f'abc -c "{abc_script}"'
    exit_code, output = run_command(command)

    return exit_code, output

def aig_to_dimacs(aig_file, output_dimacs="output.cnf"):
    """
    Convert an AIG file to DIMACS CNF using ABC.

    Args:
        aig_file (str): Input AIG file
        output_dimacs (str): Output DIMACS CNF file

    Returns:
        (exit_code, output)
    """

    # Create all leading dirs to output_aig if they don't exist
    output_dir = os.path.dirname(output_dimacs)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    abc_script = f"""
    read_aiger {aig_file};
    strash;
    write_cnf {output_dimacs}
    """

    command = f'abc -c "{abc_script}"'
    exit_code, output = run_command(command)

    return exit_code, output

def aig_to_blif(aig_file, output_blif="output.blif"):
    """
    Convert an AIG file to BLIF using ABC.

    Args:
        aig_file (str): Input AIG file
        output_blif (str): Output BLIF file

    Returns:
        (exit_code, output)
    """

    # Create all leading dirs to output_blif if they don't exist
    output_dir = os.path.dirname(output_blif)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    abc_script = f"""
    read_aiger {aig_file};
    write_blif {output_blif}
    """

    command = f'abc -c "{abc_script}"'
    exit_code, output = run_command(command)

    return exit_code, output

def aig_miter(aig1, aig2, output_aig="miter.aig"):
    """
    Generate a miter circuit from two AIG files using ABC.

    Args:
        aig1 (str): First AIG file
        aig2 (str): Second AIG file
        output_aig (str): Output miter AIG file

    Returns:
        (exit_code, output)
    """

    # Create all leading dirs to output_aig if they don't exist
    output_dir = os.path.dirname(output_aig)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    abc_script = f"""
    read_aiger {aig1};
    miter {aig2};
    write_aiger {output_aig};
    """

    command = f'abc -c "{abc_script}"'
    exit_code, output = run_command(command)

    return exit_code, output

In [9]:
def cadical_check(dimacs_file):
    """
    Check satisfiability of a DIMACS CNF file using CaDiCaL.

    Args:
        dimacs_file (str): Input DIMACS CNF file

    Returns:
        (exit_code, output)
    """

    command = f'cadical {dimacs_file}'
    exit_code, output = run_command(command)

    return exit_code, output

## Usage

In [29]:
# Generating AIG files from Verilog sources for the two multiplier modules
verilog_to_aig(data_path/"circuits/verilog/multipliers.v", "unrolled_mult", data_path/"circuits/aig/unrolled_mult.aig")
verilog_to_aig(data_path/"circuits/verilog/multipliers.v", "behavioral_mult", data_path/"circuits/aig/behavioral_mult.aig")

(0,
 '\n /----------------------------------------------------------------------------\\\n |  yosys -- Yosys Open SYnthesis Suite                                       |\n |  Copyright (C) 2012 - 2026  Claire Xenia Wolf <claire@yosyshq.com>         |\n |  Distributed under an ISC-like license, type "license" to see terms        |\n \\----------------------------------------------------------------------------/\n Yosys 0.63+89 (git sha1 de99d67bb, clang++ 18.1.8 -fPIC -O3)\n\n-- Running command `\n    read_verilog /home/krishnendu/Research/fv-invariant-mining/data/circuits/verilog/multipliers.v\n    hierarchy -check -top behavioral_mult\n    proc\n    opt\n    techmap\n    aigmap\n    write_aiger /home/krishnendu/Research/fv-invariant-mining/data/circuits/aig/behavioral_mult.aig\n    \' --\n\n1. Executing Verilog-2005 frontend: /home/krishnendu/Research/fv-invariant-mining/data/circuits/verilog/multipliers.v\nParsing Verilog input from `/home/krishnendu/Research/fv-invariant-mining/data

In [30]:
# Use ABC to convert both AIG files to BLIF
aig_to_blif(data_path/"circuits/aig/unrolled_mult.aig", data_path/"circuits/blif/unrolled_mult.blif")
aig_to_blif(data_path/"circuits/aig/behavioral_mult.aig", data_path/"circuits/blif/behavioral_mult.blif")

(0,
 '======== ABC command line "\n    read_aiger /home/krishnendu/Research/fv-invariant-mining/data/circuits/aig/behavioral_mult.aig;\n    write_blif /home/krishnendu/Research/fv-invariant-mining/data/circuits/blif/behavioral_mult.blif\n    "\n')

In [31]:
# Use ABC to form a miter circuit from the two AIG files and convert the miter to CNF
aig_miter(data_path/"circuits/aig/unrolled_mult.aig", data_path/"circuits/aig/behavioral_mult.aig", data_path/"circuits/aig/miter_mult.aig")
aig_to_dimacs(data_path/"circuits/aig/miter_mult.aig", data_path/"circuits/cnf/miter_mult.cnf")

(0,
 '======== ABC command line "\n    read_aiger /home/krishnendu/Research/fv-invariant-mining/data/circuits/aig/miter_mult.aig;\n    strash;\n    write_cnf /home/krishnendu/Research/fv-invariant-mining/data/circuits/cnf/miter_mult.cnf\n    "\nCNF stats: Vars =    873. Clauses =    3299. Literals =     9434.   Time =     0.01 sec\n')

In [32]:
# Use ABC to convert both AIG files to DIMACS CNF
aig_to_dimacs(data_path/"circuits/aig/unrolled_mult.aig", data_path/"circuits/cnf/unrolled_mult.cnf")
aig_to_dimacs(data_path/"circuits/aig/behavioral_mult.aig", data_path/"circuits/cnf/behavioral_mult.cnf")

(0,
 '======== ABC command line "\n    read_aiger /home/krishnendu/Research/fv-invariant-mining/data/circuits/aig/behavioral_mult.aig;\n    strash;\n    write_cnf /home/krishnendu/Research/fv-invariant-mining/data/circuits/cnf/behavioral_mult.cnf\n    "\nCNF stats: Vars =    468. Clauses =    1607. Literals =     4202.   Time =     0.01 sec\n')

In [33]:
# print(out)# Use ABC to perform CEC
ec, out = aig_cec(data_path/"circuits/aig/unrolled_mult.aig", data_path/"circuits/aig/behavioral_mult.aig")

In [34]:
# Use CaDiCaL to check satisfiability of the miter CNF
ec, out = cadical_check(data_path/"circuits/cnf/miter_mult.cnf")
if "UNSATISFIABLE" in out:
    print("The miter corresponds to two equivalent implementations.")
elif "SATISFIABLE" in out:
    print("The miter corresponds to two non-equivalent implementations.")

The miter corresponds to two equivalent implementations.
